# Qwen3-VL-Embedding-8B でメッセージのembeddingを生成

このノートブックでは、traQのメッセージからテキストと画像を抽出し、Qwen3-VL-Embedding-8Bモデルでembeddingを生成します。


In [ ]:
# 必要なパッケージのインストール
%pip install transformers accelerate torch pillow requests python-dotenv numpy scikit-learn matplotlib seaborn qwen-vl-utils  -q

# 公式のQwen3VLEmbedderクラスをインストール
!mkdir -p scripts
!wget -q https://huggingface.co/Qwen/Qwen3-VL-Embedding-8B/resolve/main/scripts/qwen3_vl_embedding.py -O scripts/qwen3_vl_embedding.py

In [21]:
import torch
import gc

# メモリをクリア
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


In [ ]:
import requests
import os 
import re
import io
from PIL import Image


messageIds = [
    "019b2a64-eebc-7815-84c5-e384fabc495c", #ictトラブルシューティング予選結果
    "019a48ff-b175-783c-93fc-b4ec6731dde5", #TechBookの販売記録
    "019bb6b8-5c57-7d1f-8b93-297c8e6c7827", #ポータブルモニターを購入
    "019bb5c0-5409-7b3b-ba9b-07449030364a", #大学の課題の締め切り
    "019bd043-d86c-79b8-9cf1-11351a482267", #pcの空き容量の画像
    "0197c4ef-8be2-792e-922b-c498b6948775", #traqの9点リーダーのアイコン
    "01963951-0f55-75b5-a53b-07f467ad9051", #sysad体験会
    "0195be37-dea6-7902-8d04-e46185873108", #githubへの招待
    "019bde55-0ca0-7d7a-95ab-0a1c3bf5eeaa", #僕のgithub
    "019bde55-b166-7d7b-b290-2f049f72fe08", #googleの検索画面
    "019bbd0d-5e27-77b2-8494-42499697c033", #大学の課題をやろうとしてる。reportのpdf画像　やるぞ～と言ってる
    "019bde84-1609-7dc5-b8fc-f2c77bcdb005", #食べ物の画像すき焼き弁当
]



def _get_session() -> requests.Session:
    r_session = os.getenv("r_session")
    if not r_session:
        raise RuntimeError("r_session が見つかりません（環境変数を設定してください）")
    
    session = requests.Session()
    session.cookies.set("r_session", r_session)
    return session


# messageidの配列を受け取り、メッセージの配列を返す
def get_messages_and_images():
    BASE_URL = "https://q.trap.jp/api/v3"
    _FILE_URL_RE = re.compile(r"https?://q\.trap\.jp/files/([0-9a-fA-F-]+)")
    
    messages_result = []
    images_result = []
    
    with _get_session() as session:
        for message_id in messageIds:
            
            try:
                response = session.get(f"{BASE_URL}/messages/{message_id}")
                response.raise_for_status()
                content = response.json()["content"]

                """
                画像のurlを抽出
                メッセージのcontentからファイルIDを抽出
                画像はcontentの中にhttps://q.trap.jp/files/019b2a64-ee12-7815-9d3c-4c510250218aのような形式で保存されている
                """
                image_file_ids = _FILE_URL_RE.findall(content)

                imgs = []

                for image_file_id in image_file_ids:
                    response = session.get(f"{BASE_URL}/files/{image_file_id}")
                    response.raise_for_status()
                    img_bytes = response.content
                    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                    imgs.append(pil_img)


                # content内にある 画像のurlを削除 (embeddingにurlは不要なため)
                processed_text = _FILE_URL_RE.sub("", content).strip()

                messages_result.append(processed_text)
                images_result.append(imgs)
            except requests.exceptions.RequestException as e:
                print(f"Error fetching message {message_id}: {e}")
    
    return messages_result, images_result


In [ ]:
# traQ APIのセッション情報を設定
# vscodeだと.envが使えない？
import os

r_session = input("r_sessionを入力してください: ").strip()
if r_session:
    os.environ["r_session"] = r_session
    print("r_sessionを設定しました")
else:
    print("警告: r_sessionが設定されていません")

## モデルのロード

最初にモデルをロードします。これには5.5分程度かかる場合があります。


In [ ]:
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder
import torch

model_name_or_path = "Qwen/Qwen3-VL-Embedding-8B"

model = Qwen3VLEmbedder(
    model_name_or_path=model_name_or_path,
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",
)


print("model loaded")



## メッセージの取得とembedding生成

メッセージを取得して、それぞれのembeddingを生成します。


In [ ]:
# メッセージと画像を取得
messages, images = get_messages_and_images()
print(f"取得したメッセージ数: {len(messages)}")
print(f"画像ありメッセージ数: {sum(1 for imgs in images if len(imgs) > 0)}")

# Qwen3VLEmbedder 用の入力を作成
documents = []
for idx, (text, imgs) in enumerate(zip(messages, images)):
    # 複数画像がある場合は、各画像ごとにテキスト+画像の組み合わせを作成
    for img in imgs:
        documents.append({"text": text, "image": img})

print(f"入力数: {len(documents)}")

In [ ]:
# embeddingを生成
import os
import json

os.makedirs("out", exist_ok=True)

queries = [
    "github",
    "pc",
    "モニター",
    "TechBook",
    "SysAd",
    "新入生",
    "テスト",
    "紙",
    "mumumu",
    "google",
    "googleのロゴが左上にありqwen3-vlについて検索してる",
    "大学の課題をやろうとしてる",
    "やる気がある",
    "ごはん",
    "駅弁",
    "すきやき"
]

inputs = queries + documents

print("embedding を生成中...")
embeddings = model.process(inputs)

# 出力を保存
similarity_scores = (embeddings[:4] @ embeddings[4:].T)
print(similarity_scores.tolist())
